# Step 2b — Baseline Models: Popularity, BM25 and Hybrid NCF-LSTM

## Objective

This notebook establishes reference benchmarks for our JDSearch personalised product re-ranking problem by implementing three models: popularity, BM25 and an adapted Hybrid NCF-LSTM.

The previous group's Hybrid NCF architecture primarily formulates recommendation as a four-class interaction prediction task: no interaction (0), click (1), add-to-cart (2) and purchase (3). Our implementation preserves this pointwise classification approach while adapting its inputs and evaluation to our group's re-ranking objective.

### Models implemented

1. **Popularity baseline:** Ranks candidate products using their historical popularity across training users, with a small adjustment for historical orders.
2. **BM25 baseline:** Ranks candidates using lexical relevance between the current query and product name.
3. **Hybrid NCF-LSTM:** Combines sequential user history, categorical candidate embeddings and engineered numerical features to predict interaction probabilities.

### Main adaptations

- Use our group's 50% purchase-stratified user sample and existing 70/15/15 user-disjoint split.
- Retain complete candidate slates for validation and testing.
- Preserve the previous group's two-layer LSTM and historical attention mechanism.
- Reduce the candidate encoder from seven additive embeddings to four.
- Expand engineered numerical predictors from 15 to 53.
- Convert predicted class probabilities into expected graded-relevance scores for ranking.

### Pipeline dependencies

This notebook requires the outputs of Steps 1a, 1b and 2a.

Step 1a supplies historical user, candidate and group features. Step 1b supplies query–product relevance and catalogue features. Step 2a supplies the compact metadata lookup used by the four-embedding candidate encoder.


## 1. Environment and Experiment Configuration

We initialize the Python and PyTorch environment, set random seeds and configure the computational device.

The notebook supports GPU acceleration through CUDA, which is particularly important for the LSTM's recurrent computations and candidate-level neural classification.

### Key hyperparameters

| Parameter | Value | Purpose |
|---|---|---|
| `SEED` | 42 | Reproducibility of random initialization and sampling |
| `QUICK_RUN` | False | Use the complete sampled dataset for final experiments |
| `USE_METADATA` | True | Enable the brand, shop and category embeddings |
| `MAX_HISTORY` | 200 | Maximum historical interactions processed per user |
| `MAX_VOCAB` | 250,000 | Cap product-ID vocabulary size to control memory usage |
| `EMB` | 64 | Dimensionality of learned embeddings |
| `HIDDEN` | 64 | LSTM hidden-state dimensionality |
| `BATCH` | 32 | Number of search sessions per mini-batch |
| `EPOCHS` | 12 | Maximum training epochs for the full experiment |
| `NEG_PER_POS` | 20 | Maximum negative-to-positive ratio during training |

The quick-run configuration is intended for debugging only and does not provide representative final performance estimates.

**Computational considerations:** Product-ID embeddings can be particularly memory-intensive because the catalogue contains millions of distinct products. We cap the item vocabulary and truncate sequential histories to make training feasible within Google Colab.

These constraints apply to the neural representation, not to the number of candidates evaluated within each search.

In [2]:
%pip -q install pyarrow scikit-learn

import os
import gc
import time
import random

from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from IPython.display import display

from google.colab import drive
drive.mount("/content/drive")

# ============================================================
# CONFIGURATION
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

ROOT = Path("/content/drive/MyDrive/bt4222/JDsearch")

STEP1A = ROOT / "step1a_history"
STEP1B = ROOT / "step1b_query_item" / "full"

SAMPLED = ROOT / "user_subset.parquet"
METADATA = (
    ROOT / "step2_baselines"
    / "candidate_metadata_compact.parquet"
)

OUT = ROOT / "step2_baselines"
OUT.mkdir(parents=True, exist_ok=True)

# Experiment switches
QUICK_RUN = False
USE_METADATA = True

LIMITS = {
    "train": 1000,
    "val": 300,
    "test": 300
}

MAX_HISTORY = 200
MAX_VOCAB = 250_000

EMB = 64
HIDDEN = 64
BATCH = 32

EPOCHS = 2 if QUICK_RUN else 12
NEG_PER_POS = 20

print("Device:", DEVICE)
if DEVICE != "cuda":
    print("WARNING: CPU training will be much slower. In Colab select Runtime > Change runtime type > T4 GPU, then restart.")
print("Quick run:", QUICK_RUN)
print("Metadata:", USE_METADATA)
print("Data directory:", ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
Quick run: False
Metadata: True
Data directory: /content/drive/MyDrive/bt4222/JDsearch


## 2. Session Selection and Input Files

### 2.1 Preserving the Existing Sampling Scheme

We load the sampled user dataset and preserve its original session identifiers and split assignments.

Our group's sampling strategy retains 85,864 eligible search sessions, approximately 50% of the filtered JDSearch population. Sampling is stratified by whether the session contains a purchase, preserving the purchase-session prevalence.

The existing user-disjoint 70/15/15 split is retained:

- Training: 60,104 sessions.
- Validation: 12,880 sessions.
- Testing: 12,880 sessions.

The `session_id` corresponds to the original row index in `user_subset.parquet`. Therefore, the source file must not be reordered before assigning session identifiers.


### 2.2 Input Dependencies

We load the following upstream outputs:

| Source | File | Purpose |
|---|---|---|
| Step 1a | `history_full.parquet` | Historical product, action and time sequences |
| Step 1a | `history_user_features.parquet` | User-level behavioural features |
| Step 1a | `history_candidate_features.parquet` | Exact-product interaction history |
| Step 1a | `history_group_features.parquet` | Brand, shop and category history |
| Step 1b | `query_item_train/val/test.parquet` | Query–candidate relevance and catalogue features |
| Step 2a | `candidate_metadata_compact.parquet` | Categorical metadata for candidate embeddings |


In [3]:
# ============================================================
# EARLY SESSION FILTERING
# ============================================================

source = pd.read_parquet(
    SAMPLED,
    columns=["split"]
)

# session_id = original row number
source["session_id"] = np.arange(len(source), dtype=np.int64)

source["split"] = (
    source["split"]
    .astype(str)
    .str.lower()
    .replace({"validation": "val", "valid": "val"})
)

if QUICK_RUN:
    selected = pd.concat(
        [
            source[source["split"] == s].head(LIMITS[s])
            for s in ["train", "val", "test"]
        ],
        ignore_index=True
    )
else:
    selected = source.copy()

selected_sids = set(selected["session_id"])

print("Selected sessions:", len(selected))
print(selected["split"].value_counts())

del source
gc.collect()

Selected sessions: 85864
split
train    60104
val      12880
test     12880
Name: count, dtype: int64


16

In [4]:
import pyarrow.dataset as ds

def load_sessions(path, columns=None):
    path = Path(path)

    if QUICK_RUN:
        # Arrow filters before Pandas materialization.
        dataset = ds.dataset(str(path), format="parquet")

        table = dataset.to_table(
            columns=columns,
            filter=ds.field("session_id").isin(
                sorted(selected_sids)
            )
        )

        return table.to_pandas()

    return pd.read_parquet(path, columns=columns)


paths = {
    "full": STEP1A / "history_full.parquet",
    "hu": STEP1A / "history_user_features.parquet",
    "hc": STEP1A / "history_candidate_features.parquet",
    "hg": STEP1A / "history_group_features.parquet",
    "qi_train": STEP1B / "query_item_train.parquet",
    "qi_val": STEP1B / "query_item_val.parquet",
    "qi_test": STEP1B / "query_item_test.parquet",
}

if USE_METADATA:
    paths["metadata"] = METADATA

for name, path in paths.items():
    print(name, path.exists(), path)

missing = [str(p) for p in paths.values() if not p.exists()]

if missing:
    raise FileNotFoundError("\n".join(missing))

full True /content/drive/MyDrive/bt4222/JDsearch/step1a_history/history_full.parquet
hu True /content/drive/MyDrive/bt4222/JDsearch/step1a_history/history_user_features.parquet
hc True /content/drive/MyDrive/bt4222/JDsearch/step1a_history/history_candidate_features.parquet
hg True /content/drive/MyDrive/bt4222/JDsearch/step1a_history/history_group_features.parquet
qi_train True /content/drive/MyDrive/bt4222/JDsearch/step1b_query_item/full/query_item_train.parquet
qi_val True /content/drive/MyDrive/bt4222/JDsearch/step1b_query_item/full/query_item_val.parquet
qi_test True /content/drive/MyDrive/bt4222/JDsearch/step1b_query_item/full/query_item_test.parquet
metadata True /content/drive/MyDrive/bt4222/JDsearch/step2_baselines/candidate_metadata_compact.parquet


## 3. Integrating Historical and Query–Product Features

The model combines the feature engineering outputs of Steps 1a and 1b into one candidate-level dataset.

Each observation represents a candidate product within a particular search session, identified by the composite key `(session_id, wid)`.

### 3.1 Identifier Standardization

We first standardize product identifiers using `norm_id()`.

This prevents mismatches caused by inconsistent string or numerical representations of the same product ID across intermediate files.

### 3.2 Query–Product Features

The Step 1b train, validation and test files are concatenated while preserving their existing split assignments.

These files provide the starting candidate population, including the original interaction labels and query–product features.

### 3.3 Exact-Product Historical Features

We left-join `history_candidate_features.parquet` using `(session_id, wid)`.

These features capture whether and how frequently the user previously interacted with the exact candidate product, as well as the recency of those interactions.

We compare labels and split assignments across the two sources to detect inconsistencies.

### 3.4 User-Level Historical Features

We join `history_user_features.parquet` using `session_id`, with a many-to-one relationship.

These features summarize overall historical activity, interaction composition and recency.

### 3.5 Shared-Attribute Historical Features

We join `history_group_features.parquet` using `(session_id, wid)`.

These features capture previous interactions with products sharing the candidate's brand, shop or category, allowing historical preferences to inform predictions even when the exact candidate has not been encountered.

In [5]:
# ============================================================
# JOIN FEATURE ENGINEERING OUTPUTS
# ============================================================

def norm_id(x):
    if pd.isna(x):
        return None

    s = str(x).strip()

    if s.endswith(".0") and s[:-2].isdigit():
        s = s[:-2]

    return s


def check_unique(d, keys, name):
    n = d.duplicated(keys).sum()
    print(f"{name} duplicate keys: {n:,}")


qi_parts = []

for s in ["train", "val", "test"]:
    t = load_sessions(paths[f"qi_{s}"])

    t = t.rename(columns={"candidate_id": "wid"})
    t["wid"] = t["wid"].map(norm_id)

    assert (
        t["split"].astype(str).str.lower() == s
    ).all(), f"Split mismatch in {s}"

    qi_parts.append(t)

qi = pd.concat(qi_parts, ignore_index=True)
del qi_parts

original_rows = len(qi)

check_unique(qi, ["session_id", "wid"], "Step 1b")

# ------------------------------------------------------------
# Candidate history features
# ------------------------------------------------------------

hc = load_sessions(paths["hc"])
hc["wid"] = hc["wid"].map(norm_id)

check_unique(hc, ["session_id", "wid"], "Step 1a candidate")

hc = hc.rename(columns={
    c: f"hc__{c}"
    for c in hc.columns
    if c not in ["session_id", "wid", "label", "split"]
})

df = qi.merge(
    hc,
    on=["session_id", "wid"],
    how="left",
    suffixes=("_qi", "_hc"),
    indicator=True,
    validate="one_to_one"
)

print("Candidate join unmatched:",
      (df["_merge"] != "both").sum())

print("Label mismatches:",
      ((df["label_qi"] != df["label_hc"]) &
       df["label_hc"].notna()).sum())

print("Split mismatches:",
      ((df["split_qi"] != df["split_hc"]) &
       df["split_hc"].notna()).sum())

df = (
    df.drop(columns=["_merge", "label_hc", "split_hc"])
      .rename(columns={
          "label_qi": "label",
          "split_qi": "split"
      })
)

del hc, qi
gc.collect()

# ------------------------------------------------------------
# User history features
# ------------------------------------------------------------

hu = load_sessions(paths["hu"])
check_unique(hu, ["session_id"], "Step 1a user")

hu = hu.drop(columns=["split"])

hu = hu.rename(columns={
    c: f"hu__{c}"
    for c in hu.columns
    if c != "session_id"
})

df = df.merge(
    hu,
    on="session_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("Missing user joins:",
      (df["_merge"] != "both").sum())

df = df.drop(columns="_merge")

del hu
gc.collect()

# ------------------------------------------------------------
# Brand/shop/category history features
# ------------------------------------------------------------

hg = load_sessions(paths["hg"])
hg["wid"] = hg["wid"].map(norm_id)

check_unique(hg, ["session_id", "wid"], "Step 1a group")

hg = hg.rename(columns={
    c: f"hg__{c}"
    for c in hg.columns
    if c not in ["session_id", "wid"]
})

df = df.merge(
    hg,
    on=["session_id", "wid"],
    how="left",
    validate="one_to_one",
    indicator=True
)

print("Missing group joins:",
      (df["_merge"] != "both").sum())

df = df.drop(columns="_merge")

del hg
gc.collect()

print("\nOriginal candidate rows:", original_rows)
print("Joined candidate rows:", len(df))
print("Rows lost:", original_rows - len(df))

display(
    df.groupby("split").agg(
        sessions=("session_id", "nunique"),
        candidates=("wid", "size"),
        positives=("label", lambda x: (x > 0).sum())
    )
)

Step 1b duplicate keys: 0
Step 1a candidate duplicate keys: 0
Candidate join unmatched: 0
Label mismatches: 0
Split mismatches: 0
Step 1a user duplicate keys: 0
Missing user joins: 0
Step 1a group duplicate keys: 0
Missing group joins: 0

Original candidate rows: 7665446
Joined candidate rows: 7665446
Rows lost: 0


,sessions,candidates,positives
split,,,
test,12880,1147673,23929
train,60104,5360982,111997
val,12880,1156791,24309


## 4. Data Integrity and Leakage Checks

We validate the integrated dataset before constructing model inputs.

### Candidate Preservation

The number of rows after joining must match the original Step 1b candidate count. This ensures that feature integration does not unintentionally remove or duplicate candidate observations.

### Identifier and Split Consistency

We inspect duplicate `(session_id, wid)` keys, missing product identifiers, invalid interaction labels and sessions assigned to multiple splits.

We also examine missing feature matches across the historical feature families.

### Target Leakage Prevention

Candidate position is excluded from model predictors because the stored JDSearch candidate lists are ordered in a way that reveals engagement outcomes.

Similarly, interaction labels, split identifiers and target-derived variables are excluded from the numerical feature matrix.

These checks support the validity of our offline ranking evaluation by ensuring that the model does not directly receive the outcomes it is intended to predict.

In [6]:
# ============================================================
# JOIN INTEGRITY CHECKS
# ============================================================

print("All candidate rows retained:", len(df) == original_rows)

print(
    "Sessions with multiple splits:",
    (df.groupby("session_id")["split"].nunique() > 1).sum()
)

print(
    "Invalid labels:",
    (~df["label"].isin([0, 1, 2, 3])).sum()
)

print(
    "Missing candidate IDs:",
    df["wid"].isna().sum()
)

print(
    "Duplicate session-product pairs:",
    df.duplicated(["session_id", "wid"]).sum()
)

print("\nFeature-family counts:")

for prefix in ["hc__", "hg__", "hu__"]:
    print(
        prefix,
        sum(c.startswith(prefix) for c in df.columns)
    )

All candidate rows retained: True
Sessions with multiple splits: 0
Invalid labels: 0
Missing candidate IDs: 0
Duplicate session-product pairs: 0

Feature-family counts:
hc__ 9
hg__ 15
hu__ 17


## 5. Numerical Feature Selection and Standardization

### 5.1 Feature Selection

We construct the numerical input matrix using engineered features from Steps 1a and 1b.

The intended feature families are:

| Feature family | Count | Information captured |
|---|---:|---|
| Query and product features | 12 | Query overlap, BM25 relevance and catalogue characteristics |
| Exact-product history | 9 | Previous interactions with the candidate product |
| Brand/shop/category history | 15 | Preferences for related products |
| User-level history | 17 | Overall activity, interaction composition and recency |
| **Total** | **53** | |

Only numerical columns are selected. Identifier, target and position-related columns are excluded.

The final selected feature names and count are printed for inspection.

### 5.2 Missing-Value Treatment

Missing numerical values are replaced using feature-wise medians calculated from training observations only.

This avoids allowing validation or test distributions to influence the imputation parameters.

### 5.3 Standardisation

A `StandardScaler` is fitted using training observations and subsequently applied to all splits.

Standardisation places numerical predictors on comparable scales, improving optimisation when combining heterogeneous behavioural counts, recency measures and query-relevance scores.


In [7]:
# ============================================================
# NUMERICAL FEATURE PREPARATION
# ============================================================

EXCLUDE = {
    "session_id",
    "wid",
    "label",
    "split",
    "candidate_pos",
    "candidate_id",
    "has_purchase"
}

FEATURES = []

for col in df.columns:

    if col in EXCLUDE:
        continue

    if not pd.api.types.is_numeric_dtype(df[col]):
        continue

    lc = col.lower()

    if any(x in lc for x in [
        "position",
        "rank",
        "label",
        "has_purchase",
        "target",
        "conversion_rate",
        "purchase_rate",
        "click_rate",
        "cart_rate"
    ]):
        continue

    FEATURES.append(col)

print("Numerical predictors:", len(FEATURES))
print(FEATURES)

train_mask = df["split"].eq("train")

med = (
    df.loc[train_mask, FEATURES]
      .median()
      .fillna(0)
)

scaler = StandardScaler()

scaler.fit(
    df.loc[train_mask, FEATURES]
      .fillna(med)
      .astype("float32")
)

X = np.empty(
    (len(df), len(FEATURES)),
    dtype="float32"
)

for start in range(0, len(df), 100_000):

    end = min(start + 100_000, len(df))

    block = (
        df.iloc[start:end][FEATURES]
          .fillna(med)
          .astype("float32")
    )

    X[start:end] = scaler.transform(block)

print("Feature matrix:", X.shape)
print("Memory MB:", round(X.nbytes / 1e6, 2))

assert np.isfinite(X).all()

Numerical predictors: 53
['has_metadata', 'query_name_overlap', 'query_brand_overlap', 'query_cate_overlap', 'bm25_query_name', 'name_length', 'has_brand', 'shop_is_missing', 'cate_depth', 'brand_product_count', 'shop_product_count', 'cate4_product_count', 'hc__prod_n_click', 'hc__prod_n_cart', 'hc__prod_n_follow', 'hc__prod_n_order', 'hc__prod_days_since_click', 'hc__prod_days_since_cart', 'hc__prod_days_since_order', 'hc__prod_days_since_any', 'hc__prod_seen_before', 'hu__n_click_7d', 'hu__n_cart_90d', 'hu__n_follow_90d', 'hu__n_order_rows_365d', 'hu__n_checkouts_365d', 'hu__n_order_products_365d', 'hu__has_click', 'hu__days_since_last_click', 'hu__has_cart', 'hu__days_since_last_cart', 'hu__has_order', 'hu__days_since_last_order', 'hu__days_since_last_action', 'hu__in_session', 'hu__n_actions_7d', 'hu__intent_share_7d', 'hu__history_len_full', 'hg__brand_n_click', 'hg__brand_n_cart', 'hg__brand_n_order', 'hg__brand_days_since_any', 'hg__brand_days_since_order', 'hg__shop_n_click', '

## 6. Historical Sequences and Product-ID Vocabulary

### 6.1 Historical Sequence Construction

We load the historical interaction lists from `history_full.parquet`.

For each session, we retrieve three aligned sequences:

- `history_wid_list`: Products previously interacted with.
- `history_type_list`: Historical interaction types (CLICK, CART, ORD, FLW).
- `history_time_list`: Time intervals associated with historical interactions.

The code verifies that product and action sequences have equal lengths and that the time-gap sequence contains one additional element.

### 6.2 Historical Sequence Truncation

We retain the latest 200 interactions for the LSTM:

`MAX_HISTORY = 200`

This limits recurrent sequence length and reduces computational requirements. Our interim analysis found that this cutoff preserves complete histories for approximately 76.7% of users.

However, a single recency-based cutoff may disproportionately discard older orders, given the dataset's asymmetric historical retention windows.

### 6.3 Product-ID Vocabulary Construction

We construct the product-ID vocabulary using training-user histories and training candidate identities.

Products are ranked by their frequency in this training data, and the vocabulary is capped at 250,000 entries.

Two special indices are reserved:

- `PAD_IDX = 0`: Padding for variable-length sequences.
- `UNK_IDX = 1`: Products without a dedicated vocabulary entry.

Candidate products outside the vocabulary are mapped to the shared unknown embedding.

The unknown-item rate is reported separately for each split to assess the extent of product-level sparsity and vocabulary coverage.

In [8]:
# ============================================================
# HISTORIES AND ITEM VOCABULARY
# ============================================================

full = load_sessions(
    paths["full"],
    columns=[
        "session_id",
        "split",
        "history_wid_list",
        "history_type_list",
        "history_time_list"
    ]
)

def parse_list(x):
    if isinstance(x, (list, tuple, np.ndarray)):
        return list(x)

    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []

    return str(x).split("_") if str(x) else []


H = {}

for row in full.itertuples(index=False):

    sid = int(row.session_id)

    wids = parse_list(row.history_wid_list)
    acts = parse_list(row.history_type_list)
    times = parse_list(row.history_time_list)

    if not (
        len(wids) == len(acts)
        and len(times) == len(wids) + 1
    ):
        raise ValueError(f"History alignment error: {sid}")

    H[sid] = (
        wids[-MAX_HISTORY:],
        acts[-MAX_HISTORY:],
        times[-(MAX_HISTORY + 1):]
    )

print("Loaded histories:", len(H))

train_sids = set(
    selected.loc[
        selected["split"] == "train",
        "session_id"
    ]
)

# POPULARITY BASELINE

from collections import Counter

HISTORY_LONG = (
    ROOT / "step1a_history" / "history_long.parquet"
)

hist = pd.read_parquet(
    HISTORY_LONG,
    columns=["session_id", "wid", "type", "age_sec"]
)

# Training users only
hist = hist[hist["session_id"].isin(train_sids)].copy()

# Exclude historical actions within 1 second of the target search
hist = hist[hist["age_sec"] > 1].copy()

hist["wid"] = hist["wid"].map(norm_id)
hist["type"] = hist["type"].astype(str)

hist = hist.dropna(subset=["wid"])

# ------------------------------------------------------------
# 1. Distinct training-user popularity
# ------------------------------------------------------------

user_product = hist[
    ["session_id", "wid"]
].drop_duplicates()

pop_product_users = Counter(
    user_product["wid"].value_counts().to_dict()
)

# ------------------------------------------------------------
# 2. Historical order frequency
# ------------------------------------------------------------

orders = hist[hist["type"] == "ORD"]

pop_product_ord = Counter(
    orders["wid"].value_counts().to_dict()
)

# ------------------------------------------------------------
# 3. Own-history contributions for training evaluation
# ------------------------------------------------------------

own_products = set(
    zip(user_product["session_id"], user_product["wid"])
)

own_orders = Counter(
    zip(orders["session_id"], orders["wid"])
)

print("Training history rows:", f"{len(hist):,}")
print("Products with popularity:", f"{len(pop_product_users):,}")
print("Products with orders:", f"{len(pop_product_ord):,}")

del hist, user_product, orders
gc.collect()

# Historical interaction popularity
popularity_counts = Counter()

for sid in train_sids:
    popularity_counts.update(
        norm_id(w) for w in H[int(sid)][0]
    )

# Item vocabulary based on training histories and
# training candidate identities
vocab_counts = Counter(popularity_counts)

vocab_counts.update(
    df.loc[df["split"] == "train", "wid"].tolist()
)

vocab = {
    w: i + 2
    for i, (w, _) in enumerate(
        (pair for pair in vocab_counts.most_common()
         if pair[0] is not None)
    )
    if i < MAX_VOCAB - 2
}

PAD_IDX = 0
UNK_IDX = 1

ACT = {
    "CLICK": 1,
    "CART": 2,
    "ORD": 3,
    "FLW": 4
}

item_idx = (
    df["wid"]
    .map(lambda w: vocab.get(w, UNK_IDX))
    .to_numpy(dtype="int32")
)

print("Item vocabulary:", len(vocab))

for s in ["train", "val", "test"]:
    mask = df["split"].eq(s).to_numpy()

    print(
        s,
        "Unknown item rate:",
        round((item_idx[mask] == UNK_IDX).mean(), 5)
    )

del full
gc.collect()

Loaded histories: 85864
Training history rows: 9,218,252
Products with popularity: 2,580,944
Products with orders: 934,838
Item vocabulary: 249998
train Unknown item rate: 0.79314
val Unknown item rate: 0.81516
test Unknown item rate: 0.8135


0

## 7. Candidate Metadata Encoding

### 7.1 Metadata Source

We load the compact product-metadata lookup prepared in Step 2a.

The file contains the product identifiers and three categorical attributes used alongside the product-ID embedding:

- Brand ID.
- Shop ID.
- Level-4 category ID.


### 7.2 Categorical Vocabulary Construction

For each metadata attribute, we construct a mapping from categorical identifiers to integer embedding indices.

The resulting candidate-attribute matrix has three columns, corresponding to brand, shop and level-4 category.

The product-ID embedding is handled separately through the vocabulary constructed in Section 6.

### 7.3 Adaptation from the Previous Architecture

The previous group's candidate encoder used seven additive embeddings: product ID, brand, shop and category levels 1–4.

Our adapted model retains four.

Each embedding has 64 dimensions. Their element-wise sum produces a single 64-dimensional candidate representation.

This reduces embedding-table requirements while retaining shared product-attribute information.

Such shared embeddings are especially useful when individual products have limited historical interactions, although they do not completely resolve cold-item sparsity.

### 7.4 Metadata Coverage Diagnostics

We report the number of matched products, categorical vocabulary sizes and unknown-attribute rates.

These diagnostics help assess whether the available metadata can support generalisation beyond individually learned product IDs.

In [9]:
# ============================================================
# FOUR-EMBEDDING METADATA LOOKUP
# ============================================================

ATTR_COLS = [
    "brand_id",
    "shop_id",
    "cate_id_4"
]

candidate_attrs = np.ones(
    (len(df), len(ATTR_COLS)),
    dtype=np.int32
)

attr_card = {}

if USE_METADATA:

    required_wids = set(df["wid"].dropna())

    pf = pq.ParquetFile(METADATA)

    meta_parts = []

    for batch in pf.iter_batches(
        batch_size=100_000,
        columns=["wid"] + ATTR_COLS
    ):

        part = batch.to_pandas()

        part["wid"] = part["wid"].map(norm_id)

        part = part[
            part["wid"].isin(required_wids)
        ]

        if not part.empty:
            meta_parts.append(part)

        del part

    if meta_parts:
        meta = pd.concat(meta_parts, ignore_index=True)
    else:
        meta = pd.DataFrame(columns=["wid"] + ATTR_COLS)

    duplicate_meta = meta.duplicated("wid").sum()

    print("Duplicate metadata WIDs:", duplicate_meta)

    meta = meta.drop_duplicates("wid")

    print("Matched metadata products:", len(meta))
    print("Candidate products:", len(required_wids))

    # Build categorical lookup tables
    for j, col in enumerate(ATTR_COLS):

        vals = meta[col].map(norm_id)

        valid = (
            vals.notna()
            & ~vals.isin(["-1", "", "nan", "None"])
        )

        categories = pd.Index(vals[valid].unique())

        mapping = {
            v: i + 2
            for i, v in enumerate(categories)
        }

        attr_card[col] = len(mapping) + 2

        lookup = pd.Series(
            vals.map(
                lambda v: mapping.get(v, UNK_IDX)
            ).to_numpy(dtype="int32"),
            index=meta["wid"]
        )

        candidate_attrs[:, j] = (
            df["wid"]
            .map(lookup)
            .fillna(UNK_IDX)
            .to_numpy(dtype="int32")
        )

        del lookup, mapping, categories

    print("Attribute vocabulary sizes:", attr_card)

    for j, col in enumerate(ATTR_COLS):
        print(
            col,
            "Unknown rate:",
            round(
                (candidate_attrs[:, j] == UNK_IDX).mean(),
                5
            )
        )

    del meta, meta_parts, required_wids
    gc.collect()

else:

    print("Metadata disabled: item-ID embeddings only")

Duplicate metadata WIDs: 0
Matched metadata products: 4829298
Candidate products: 4859472
Attribute vocabulary sizes: {'brand_id': 145357, 'shop_id': 219070, 'cate_id_4': 6505}
brand_id Unknown rate: 0.00456
shop_id Unknown rate: 0.01423
cate_id_4 Unknown rate: 0.00456


## 8. Ranking Evaluation Framework

The original Hybrid NCF architecture predicts interaction classes independently for each user–candidate pair. Our business objective instead requires ranking candidate products within each search.

We therefore evaluate the models using search-level ranking metrics.

### 8.1 Graded Relevance

Candidate interaction labels are ordered according to engagement depth:

| Label | Interaction | NDCG gain |
|---|---|---:|
| 0 | No recorded interaction | 0 |
| 1 | Click | 1 |
| 2 | Add to cart | 3 |
| 3 | Purchase | 7 |

NDCG uses the exponential gain function \(2^y-1\), giving greater importance to correctly ranking stronger engagement outcomes.

### 8.2 Evaluation Metrics

**NDCG@5 and NDCG@10:** Measure how effectively the model places highly relevant products near the top of each search, relative to the ideal ordering.

**Mean Reciprocal Rank (MRR):** Measures how early the first positively engaged product appears.

**Purchase Hit@10:** Measures the proportion of purchase-containing searches in which at least one purchased product appears among the top ten recommendations.

### 8.3 Ranking Protocol

Candidates are ranked independently within each `session_id`.

The evaluation uses deterministic random tie-breaking, with seed 0. Candidate positions are used only to establish a reproducible ordering before tie values are assigned; they are not used as model relevance scores.

The same evaluation function is applied to popularity, BM25 and the neural model, ensuring consistent metric definitions and candidate pools.

All ranking metrics are calculated on complete candidate slates rather than the negative-sampled training batches.

In [10]:
# ============================================================
# RANKING METRICS
# ============================================================

GROUP_KEY = "session_id"
LABEL = "label"
EVAL_K = [5, 10]

def rank_metrics(d, score, ks=EVAL_K, seed=0):

    t = pd.DataFrame({
        "g": d[GROUP_KEY].to_numpy(),
        "y": d[LABEL].to_numpy(),
        "s": np.asarray(score, dtype=np.float64),
        "pos": d["candidate_pos"].to_numpy()
    })

    if t["pos"].isna().any():
        raise ValueError("Missing candidate positions")

    # Original candidate order for reproducible tie-breaking
    t = t.sort_values(
        ["g", "pos"],
        kind="stable"
    ).reset_index(drop=True)

    rng = np.random.default_rng(seed)
    t["tie"] = rng.random(len(t))

    # Rank by model score, then random tie value
    t = t.sort_values(
        ["g", "s", "tie"],
        ascending=[True, False, True],
        kind="stable"
    )

    t["r"] = t.groupby("g").cumcount() + 1
    t["gain"] = 2.0 ** t["y"] - 1
    t["disc"] = 1 / np.log2(t["r"] + 1)

    ideal = t.sort_values(
        ["g", "y"],
        ascending=[True, False],
        kind="stable"
    ).copy()

    ideal["r"] = ideal.groupby("g").cumcount() + 1
    ideal["disc"] = 1 / np.log2(ideal["r"] + 1)

    out = {}

    for k in ks:

        dcg = (
            (t["gain"] * t["disc"])
            .where(t["r"] <= k, 0)
            .groupby(t["g"])
            .sum()
        )

        idcg = (
            (ideal["gain"] * ideal["disc"])
            .where(ideal["r"] <= k, 0)
            .groupby(ideal["g"])
            .sum()
        )

        out[f"ndcg@{k}"] = (dcg / idcg).mean()

    first = (
        t[t["y"] > 0]
        .groupby("g")["r"]
        .min()
    )

    out["mrr"] = (1 / first).mean()

    buy = t[t["y"] == 3]

    if len(buy):

        out["purchase_hit@10"] = (
            buy.groupby("g")["r"].min() <= 10
        ).mean()

        out["n_purchase_searches"] = buy["g"].nunique()

    out["n_searches"] = t["g"].nunique()

    return out

## 9. Neural Training Dataset and Mini-Batch Construction

### 9.1 Pre-encoding Historical Sequences

To reduce repeated preprocessing during training, we convert historical product IDs, action types and time gaps into numerical arrays once.

Product IDs are mapped to vocabulary indices, action types to categorical indices, and time gaps are transformed using `log1p` to reduce the influence of extremely large intervals.

The encoded sequences are stored by session ID for efficient retrieval during training.

### 9.2 Session-Level Dataset

Each dataset observation corresponds to one search session, containing:

- The user's encoded historical interaction sequence.
- The session's candidate products.
- Candidate metadata indices.
- Engineered numerical features.
- Observed interaction labels.

This structure allows the LSTM to encode user history once per session, rather than independently for every candidate.

### 9.3 Training-Only Negative Subsampling

The training dataset retains all positively labelled candidates and samples up to 20 negative candidates per positive.

This reduces computational requirements and limits the dominance of label-0 observations during training.

Validation and test datasets use complete candidate slates, ensuring that evaluation reflects the original within-search ranking difficulty.

The resulting training objective is optimised on a sampled candidate distribution, while performance is assessed on the full candidate distribution.

### 9.4 Dynamic Padding and Length Bucketing

The `collate()` function pads historical sequences and candidate slates to the maximum lengths within each mini-batch.

Padding labels are assigned `-100`, allowing them to be excluded from cross-entropy loss.

The custom `BucketBatchSampler` groups sessions with similar history lengths and candidate counts, reducing unnecessary padding and improving computational efficiency.

### 9.5 DataLoader Configuration

Mini-batches contain 32 search sessions.

Training batches are shuffled, while validation and test batches retain deterministic processing.

GPU memory transfer is supported through pinned memory when CUDA is available.

In [11]:
# ============================================================
# PRE-ENCODE HISTORIES AND CANDIDATE TARGETS ONCE
# Avoid repeated pandas .iloc and string conversions in every batch.
# ============================================================
import math
from torch.utils.data import Sampler

labels_np = df["label"].to_numpy(dtype=np.int64, copy=True)
wid_np = df["wid"].to_numpy(copy=False)
# All histories are already capped at MAX_HISTORY in H.
H_encoded = {}
for sid, (wids, acts, times) in H.items():
    ww = np.fromiter((vocab.get(norm_id(w), UNK_IDX) for w in wids), dtype=np.int64, count=len(wids))
    aa = np.fromiter((ACT.get(str(a), 0) for a in acts), dtype=np.int64, count=len(acts))
    tt = np.asarray(times[:-1], dtype=np.float32)
    if len(tt) != len(ww):
        raise ValueError(f"History alignment failure for {sid}")
    tt = np.log1p(np.maximum(tt, 0))
    H_encoded[int(sid)] = (ww, aa, tt)

session_rows = {s: [] for s in ["train", "val", "test"]}
for sid, inds in df.groupby("session_id", sort=False).indices.items():
    ix = np.asarray(inds, dtype=np.int32)
    session_rows[str(df["split"].iat[int(ix[0])])].append((int(sid), ix))
for s in session_rows:
    print(s, "sessions", len(session_rows[s]), "candidate rows", sum(len(ix) for _, ix in session_rows[s]))

class Sessions(Dataset):
    def __init__(self, split, full_slate=False):
        self.split=split
        self.rows=session_rows[split]
        self.full_slate=full_slate
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        sid, idx=self.rows[i]
        if self.split == "train" and not self.full_slate and NEG_PER_POS is not None:
            y=labels_np[idx]
            pos=idx[y>0]; neg=idx[y==0]
            # Independent reproducible negative draw per access; no pandas indexing.
            take=min(len(neg), max(1,len(pos))*NEG_PER_POS)
            if take < len(neg):
                chosen=np.random.choice(neg,size=take,replace=False)
                idx=np.concatenate((pos,chosen))
        return sid,idx,*H_encoded[sid]

def collate(batch):
    B=len(batch);T=max(1,max(len(v[2]) for v in batch));K=max(len(v[1]) for v in batch)
    w=np.zeros((B,T),np.int64);a=np.zeros((B,T),np.int64);gap=np.zeros((B,T),np.float32)
    cand=np.zeros((B,K),np.int64);attrs=np.zeros((B,K,len(ATTR_COLS)),np.int64)
    feat=np.zeros((B,K,X.shape[1]),np.float32);lab=np.full((B,K),-100,np.int64)
    idxs=[];sids=[]
    for j,(sid,idx,ww,aa,tt) in enumerate(batch):
        sids.append(sid);idxs.append(idx);L=len(ww);N=len(idx)
        if L:w[j,:L]=ww;a[j,:L]=aa;gap[j,:L]=tt
        cand[j,:N]=item_idx[idx];attrs[j,:N]=candidate_attrs[idx];feat[j,:N]=X[idx];lab[j,:N]=labels_np[idx]
    return {k:torch.from_numpy(v) for k,v in dict(hist_wid=w,hist_act=a,hist_gap=gap,cand=cand,attrs=attrs,feat=feat,label=lab).items()},idxs,sids

class BucketBatchSampler(Sampler):
    # Group sessions of similar history/candidate length to reduce padding.
    def __init__(self, dataset, batch_size, shuffle=False):
        self.ds=dataset;self.bs=batch_size;self.shuffle=shuffle
    def __iter__(self):
        ids=np.arange(len(self.ds))
        if self.shuffle:np.random.shuffle(ids)
        pool=max(self.bs*40,self.bs)
        for st in range(0,len(ids),pool):
            block=ids[st:st+pool]
            block=sorted(block,key=lambda j:(len(H_encoded[self.ds.rows[j][0]][0]),len(self.ds.rows[j][1])))
            batches=[block[k:k+self.bs] for k in range(0,len(block),self.bs)]
            if self.shuffle:random.shuffle(batches)
            for b in batches:yield b
    def __len__(self):return math.ceil(len(self.ds)/self.bs)

def loader(split,shuffle=False,full_slate=False):
    dataset=Sessions(split,full_slate=full_slate)
    return DataLoader(dataset,batch_sampler=BucketBatchSampler(dataset,BATCH,shuffle),collate_fn=collate,num_workers=0,pin_memory=(DEVICE=='cuda'))

print("Pre-encoded histories:",len(H_encoded),"| batch size:",BATCH)


train sessions 60104 candidate rows 5360982
val sessions 12880 candidate rows 1156791
test sessions 12880 candidate rows 1147673
Pre-encoded histories: 85864 | batch size: 32


## 10. Hybrid NCF-LSTM Architecture

The adapted model combines three complementary representations: historical user preferences, candidate-product attributes and engineered numerical features.

### 10.1 Historical User Encoder

Each historical event is represented by the sum of three 64-dimensional components:

\[
x_t=E_{\mathrm{item}}(i_t)+E_{\mathrm{action}}(a_t)+W_g\log(1+\Delta t_t)+b_g
\]

The resulting sequence is processed by a two-layer LSTM with 64 hidden units per layer and dropout of 0.3 between recurrent layers.

The model uses packed sequences to avoid unnecessary recurrent computation on padded timesteps.

An attention layer assigns weights to historical hidden states and computes a weighted 64-dimensional user representation.

The attention weights depend on the historical hidden states, but are not explicitly conditioned on the current search query or candidate product.

### 10.2 Candidate Encoder

Candidate products are represented through four additive 64-dimensional embeddings: product ID, brand, shop and level-4 category.

A linear projection transforms the combined embedding into the candidate representation.

### 10.3 Numerical Feature Encoder

The 53 standardized numerical predictors are passed through a linear layer followed by ReLU activation, producing a 64-dimensional feature representation.

### 10.4 Feature Fusion

The three representations are concatenated, producing a 192-dimensional vector for each candidate.

### 10.5 Classification Head

The fused representation passes through a multilayer perceptron

The hidden layers use ReLU activation, batch normalization and dropout of 0.3.

The final layer outputs four logits corresponding to no interaction, click, add-to-cart and purchase.

The model is an NCF-style feature-fusion classifier rather than canonical NeuMF, which uses separate GMF and MLP pathways.

In [12]:
# ============================================================
# HYBRID NCF-LSTM MODEL
# ============================================================

class HybridNCFLSTM(nn.Module):

    def __init__(
        self,
        n_items,
        n_feat,
        attribute_cardinalities
    ):
        super().__init__()

        self.item = nn.Embedding(
            n_items, EMB, padding_idx=0
        )

        self.action = nn.Embedding(
            5, EMB, padding_idx=0
        )

        self.time = nn.Linear(1, EMB)

        self.metadata_embs = nn.ModuleList([
            nn.Embedding(n, EMB, padding_idx=0)
            for n in attribute_cardinalities
        ])

        self.lstm = nn.LSTM(
            EMB,
            HIDDEN,
            num_layers=2,
            batch_first=True,
            dropout=0.3
        )

        self.att = nn.Linear(HIDDEN, 1)

        self.candidate_proj = nn.Linear(
            EMB, HIDDEN
        )

        self.numeric = nn.Sequential(
            nn.Linear(n_feat, HIDDEN),
            nn.ReLU()
        )

        self.head = nn.Sequential(
            nn.Linear(3 * HIDDEN, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.3),

            nn.Linear(64, 4)
        )

    def forward(self, b):

        w = b["hist_wid"]
        a = b["hist_act"]
        g = b["hist_gap"]

        mask = w.ne(0)

        z = (
            self.item(w)
            + self.action(a)
            + self.time(g.unsqueeze(-1))
        )

        z = z * mask.unsqueeze(-1)

        lengths = mask.sum(1).clamp(min=1).cpu()

        packed = nn.utils.rnn.pack_padded_sequence(
            z,
            lengths,
            batch_first=True,
            enforce_sorted=False
        )

        enc, _ = self.lstm(packed)

        enc, _ = nn.utils.rnn.pad_packed_sequence(
            enc,
            batch_first=True,
            total_length=w.size(1)
        )

        att = self.att(enc).squeeze(-1)

        att = att.masked_fill(~mask, -1e9)

        att = torch.where(
            mask.any(1, keepdim=True),
            att,
            torch.zeros_like(att)
        )

        alpha = torch.softmax(att, dim=1) * mask.float()

        alpha = alpha / alpha.sum(
            1, keepdim=True
        ).clamp_min(1e-8)

        user = (
            enc * alpha.unsqueeze(-1)
        ).sum(1)

        # Candidate embedding
        cand_emb = self.item(b["cand"])

        for j, emb in enumerate(self.metadata_embs):
            cand_emb = (
                cand_emb
                + emb(b["attrs"][..., j])
            )

        c = self.candidate_proj(cand_emb)
        f = self.numeric(b["feat"])

        h = torch.cat([
            user[:, None, :].expand_as(c),
            c,
            f
        ], dim=-1)

        # Only genuine candidates enter BatchNorm.
        valid = b["label"].ne(-100)

        output = h.new_zeros(
            (*valid.shape, 4)
        )

        output[valid] = self.head(h[valid])

        return output


model = HybridNCFLSTM(
    len(vocab) + 2,
    len(FEATURES),
    [attr_card[c] for c in ATTR_COLS]
    if USE_METADATA else []
).to(DEVICE)

print("Metadata enabled:", USE_METADATA)

print(
    "Trainable parameters:",
    f"{sum(p.numel() for p in model.parameters()):,}"
)

Metadata enabled: True
Trainable parameters: 39,847,941


## 11. Model Training and Validation

### 11.1 Training Objective

The neural model is trained using four-class categorical cross-entropy loss.

Each candidate contributes a classification error based on the probability assigned to its observed interaction class.


Although candidates are grouped by search session during batching, the loss remains pointwise rather than explicitly comparing candidate relevance within the same search.

### 11.2 Optimisation

We use the Adam optimiser.

Gradient clipping limits the norm of backpropagated gradients, helping stabilize recurrent-network training.

### 11.3 Converting Classification Outputs to Ranking Scores

The model's logits are transformed into class probabilities using softmax.

We then compute the expected graded-relevance score.

This allows candidate products to be ranked by expected interaction strength, giving purchases greater weight than carts and clicks.

However, this is a post-prediction ranking transformation. The model is still trained using cross-entropy rather than a ranking-sensitive objective.

### 11.4 Validation and Early Stopping

At the end of each epoch, the model is evaluated on the complete validation candidate slates.

The checkpoint with the highest validation NDCG@10 is saved.

Training stops early if validation NDCG@10 fails to improve for three consecutive epochs.

This aligns model selection with ranking performance while preserving the original pointwise classification training objective.

### 11.5 Saved Outputs

The training process saves:

- The best model checkpoint.
- The selected epoch.
- The numerical feature list and vocabulary information.
- Epoch-level training loss and validation performance.

The checkpoint is subsequently reloaded for final evaluation.

In [13]:
# ============================================================
# TRAINING AND VALIDATION
# ============================================================

def score_from_logits(logits):

    values = torch.tensor(
        [0., 1., 2., 3.],
        device=logits.device
    )

    probs = torch.softmax(logits, dim=-1)

    return (probs * values).sum(-1)


def evaluate(split, return_predictions=False):

    model.eval()

    scores = []
    indices = []

    correct = Counter()
    total = Counter()

    with torch.no_grad():

        for batch, ixs, _ in loader(
            split, full_slate=True
        ):

            b = {
                k: v.to(DEVICE)
                for k, v in batch.items()
            }

            logits = model(b)

            ss = score_from_logits(
                logits
            ).cpu().numpy()

            pred = logits.argmax(-1).cpu().numpy()
            y = batch["label"].numpy()

            for j, ix in enumerate(ixs):

                n = len(ix)

                scores.extend(ss[j, :n])
                indices.extend(ix)

                for true, guess in zip(
                    y[j, :n], pred[j, :n]
                ):
                    total[int(true)] += 1
                    correct[int(true)] += int(true == guess)

    d = (
        df.iloc[indices][
            ["session_id", "wid", "label", "split", "candidate_pos"]
        ]
        .copy()
        .reset_index(drop=True)
    )

    sc = np.asarray(scores)

    assert len(d) == len(sc)
    assert np.isfinite(sc).all()

    m = rank_metrics(d, sc)

    m.update({
        f"class_{k}_recall":
        correct[k] / total[k] if total[k] else np.nan
        for k in range(4)
    })

    return (m, d, sc) if return_predictions else m


optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-5
)

best = -np.inf
log = []
bad = 0
patience = 3

CHECKPOINT = (
    OUT / (
        "best_lstm_ncf_4emb.pt"
        if USE_METADATA
        else "best_lstm_ncf_id_only.pt"
    )
)

for epoch in range(1, EPOCHS + 1):

    model.train()

    loss_sum = 0.
    n = 0
    t0 = time.time()

    for batch, _, _ in loader(
        "train", shuffle=True
    ):

        b = {
            k: v.to(DEVICE)
            for k, v in batch.items()
        }

        optimizer.zero_grad(set_to_none=True)

        logits = model(b)

        loss = nn.functional.cross_entropy(
            logits.reshape(-1, 4),
            b["label"].reshape(-1),
            ignore_index=-100
        )

        loss.backward()

        nn.utils.clip_grad_norm_(
            model.parameters(), 1.0
        )

        optimizer.step()

        valid = (
            b["label"] != -100
        ).sum().item()

        loss_sum += loss.item() * valid
        n += valid

    vm = evaluate("val")

    row = {
        "epoch": epoch,
        "train_ce": loss_sum / max(1, n),
        **{
            "val_" + k: v
            for k, v in vm.items()
        }
    }

    log.append(row)

    print(
        "Epoch:", epoch,
        "| Train CE:", round(row["train_ce"], 5),
        "| Val NDCG@10:", round(vm["ndcg@10"], 5),
        "| Minutes:", round((time.time()-t0)/60, 2)
    )

    if vm["ndcg@10"] > best + 1e-6:

        best = vm["ndcg@10"]
        bad = 0

        torch.save({
            "model": model.state_dict(),
            "epoch": epoch,
            "features": FEATURES,
            "vocab_size": len(vocab),
            "metadata": USE_METADATA
        }, CHECKPOINT)

    else:

        bad += 1

        if bad >= patience:
            print("Early stopping")
            break

pd.DataFrame(log).to_csv(
    OUT / "baseline_training_history.csv",
    index=False
)

print("Best validation NDCG@10:", best)

Epoch: 1 | Train CE: 0.29988 | Val NDCG@10: 0.30722 | Minutes: 1.3
Epoch: 2 | Train CE: 0.22755 | Val NDCG@10: 0.33291 | Minutes: 1.31
Epoch: 3 | Train CE: 0.22229 | Val NDCG@10: 0.34662 | Minutes: 1.29
Epoch: 4 | Train CE: 0.21861 | Val NDCG@10: 0.3531 | Minutes: 1.29
Epoch: 5 | Train CE: 0.21423 | Val NDCG@10: 0.35746 | Minutes: 1.3
Epoch: 6 | Train CE: 0.20903 | Val NDCG@10: 0.35455 | Minutes: 1.3
Epoch: 7 | Train CE: 0.20298 | Val NDCG@10: 0.34837 | Minutes: 1.26
Epoch: 8 | Train CE: 0.19619 | Val NDCG@10: 0.34345 | Minutes: 1.26
Early stopping
Best validation NDCG@10: 0.35746076665883614


## 12. Final Baseline Evaluation and Comparison

After training, we reload the best validation-selected Hybrid NCF-LSTM checkpoint.

We then evaluate three models on the same candidate pools and with the same ranking metrics.

### 12.1 Popularity Baseline

Candidates are ranked using distinct-user historical popularity, with a small historical-order adjustment.

Training-session evaluation applies leave-self-out correction, while validation and test use popularity calculated from training histories only.

### 12.2 BM25 Baseline

Candidates are ranked using the BM25 query–product-name similarity score engineered in Step 1b.

This provides a non-personalized lexical relevance benchmark against which to evaluate the value of historical behaviour and richer product representations.

### 12.3 Hybrid NCF-LSTM

The neural model generates four-class interaction logits for each candidate.

These are converted into expected graded-relevance scores and used to rank candidates within each search.

### 12.4 Consistent Evaluation

All three models are evaluated using full candidate slates and common metrics:

- NDCG@5.
- NDCG@10.
- MRR.
- Purchase Hit@10.

The evaluation covers training, validation and test splits, allowing comparison of in-sample performance and generalization to held-out users.


In [14]:
# ============================================================
# FINAL RANKING EVALUATION
# ============================================================

checkpoint = torch.load(
    CHECKPOINT,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(checkpoint["model"])

results = []

bm25_col = next(
    c for c in df.columns
    if "bm25_query_name" in c.lower()
)

bm25_lookup = df[["session_id", "wid", bm25_col]].drop_duplicates(["session_id", "wid"])

for s in ["val", "test"]:

    nn_metrics, d, nn_scores = evaluate(
        s, return_predictions=True
    )

    # DISTINCT-USER POPULARITY + ORDER TIE-BREAK

    pop = np.empty(len(d), dtype=np.float64)

    for j, (sid, wid) in enumerate(
        zip(d["session_id"], d["wid"])
    ):

        sid = int(sid)
        wid = norm_id(wid)

        n_users = pop_product_users.get(wid, 0)
        n_orders = pop_product_ord.get(wid, 0)

        # Leave-self-out for training sessions only
        if s == "train":
            n_users -= int((sid, wid) in own_products)
            n_orders -= own_orders.get((sid, wid), 0)

        pop[j] = n_users + 0.001 * n_orders
    # Lookup once via original row order when possible; preserve key validation.
    d = d.merge(bm25_lookup, on=["session_id", "wid"], how="left", validate="one_to_one")

    bm25 = (
        d[bm25_col]
        .fillna(0)
        .to_numpy()
    )

    for name, sc in [
        ("popularity", pop),
        ("bm25", bm25),
        ("lstm_ncf_ce", nn_scores)
    ]:

        results.append({
            "model": name,
            "split": s,
            **rank_metrics(d, sc)
        })

    if s == "test":

        pred = d[
            ["session_id", "wid", "label", "split"]
        ].copy()

        pred["score_lstm_ncf"] = nn_scores
        pred["score_popularity"] = pop
        pred["score_bm25"] = bm25

        pred.to_parquet(
            OUT / "test_predictions_baselines.parquet",
            index=False
        )

        print(
            "Test classifier recall:",
            {
                k: v for k, v in nn_metrics.items()
                if "recall" in k
            }
        )

res = pd.DataFrame(results)

print("\n=== ALL RANKING RESULTS ===")
display(res)

print("\n=== TEST SET PERFORMANCE ===")

display(
    res.loc[
        res["split"] == "test",
        [
            "model",
            "ndcg@5",
            "ndcg@10",
            "mrr",
            "purchase_hit@10",
            "n_searches"
        ]
    ].round(5)
)

res.to_csv(
    OUT / "results_baselines.csv",
    index=False
)

print("\nOutputs saved to:", OUT)

Test classifier recall: {'class_0_recall': 0.9996093416294103, 'class_1_recall': 0.008956127080181543, 'class_2_recall': 0.004621704895583704, 'class_3_recall': 0.004481434058898848}

=== ALL RANKING RESULTS ===


,model,split,ndcg@5,ndcg@10,mrr,purchase_hit@10,n_purchase_searches,n_searches
0,popularity,val,0.278482,0.326652,0.326130,0.574524,1523,12880
1,bm25,val,0.066023,0.091232,0.107293,0.156927,1523,12880
2,lstm_ncf_ce,val,0.311692,0.357461,0.370282,0.573867,1523,12880
3,popularity,test,0.280941,0.328184,0.325412,0.567301,1523,12880
4,bm25,test,0.064983,0.089988,0.107077,0.151018,1523,12880
5,lstm_ncf_ce,test,0.310538,0.353925,0.361620,0.556139,1523,12880



=== TEST SET PERFORMANCE ===


,model,ndcg@5,ndcg@10,mrr,purchase_hit@10,n_searches
3,popularity,0.28094,0.32818,0.32541,0.56730,12880
4,bm25,0.06498,0.08999,0.10708,0.15102,12880
5,lstm_ncf_ce,0.31054,0.35393,0.36162,0.55614,12880



Outputs saved to: /content/drive/MyDrive/bt4222/JDsearch/step2_baselines


Train NDCG is obtained separately due to heavy computation.

In [15]:
# ============================================================
# SEPARATE TRAIN NDCG@10 EVALUATION
# Memory-efficient: process complete sessions in chunks
# ============================================================

import gc

# Ensure best checkpoint is loaded
checkpoint = torch.load(
    CHECKPOINT,
    map_location=DEVICE,
    weights_only=False
)
model.load_state_dict(checkpoint["model"])
model.eval()

CHUNK_SESSIONS = 500

train_dataset = Sessions("train", full_slate=True)

weighted_ndcg = 0.0
total_sessions = 0

for start in range(0, len(train_dataset), CHUNK_SESSIONS):

    end = min(start + CHUNK_SESSIONS, len(train_dataset))

    chunk_scores = []
    chunk_indices = []

    with torch.no_grad():

        for i in range(start, end, BATCH):

            batch_items = [
                train_dataset[j]
                for j in range(i, min(i + BATCH, end))
            ]

            batch, ixs, _ = collate(batch_items)

            b = {
                k: v.to(DEVICE)
                for k, v in batch.items()
            }

            logits = model(b)

            scores = score_from_logits(logits).cpu().numpy()

            for j, ix in enumerate(ixs):
                n = len(ix)
                chunk_scores.extend(scores[j, :n])
                chunk_indices.extend(ix)

    d_chunk = (
        df.iloc[chunk_indices][
            ["session_id", "wid", "label", "split", "candidate_pos"]
        ]
        .copy()
        .reset_index(drop=True)
    )

    metrics = rank_metrics(
        d_chunk,
        np.asarray(chunk_scores)
    )

    n_sessions = metrics["n_searches"]

    weighted_ndcg += metrics["ndcg@10"] * n_sessions
    total_sessions += n_sessions

    del chunk_scores, chunk_indices, d_chunk, metrics
    gc.collect()

    if start % 5000 == 0:
        print(f"Processed {end:,} / {len(train_dataset):,} sessions")

train_ndcg10 = weighted_ndcg / total_sessions

print("\n=== FULL-SLATE TRAIN PERFORMANCE ===")
print("Train NDCG@10:", round(train_ndcg10, 6))
print("Sessions evaluated:", total_sessions)

Processed 500 / 60,104 sessions
Processed 5,500 / 60,104 sessions
Processed 10,500 / 60,104 sessions
Processed 15,500 / 60,104 sessions
Processed 20,500 / 60,104 sessions
Processed 25,500 / 60,104 sessions
Processed 30,500 / 60,104 sessions
Processed 35,500 / 60,104 sessions
Processed 40,500 / 60,104 sessions
Processed 45,500 / 60,104 sessions
Processed 50,500 / 60,104 sessions
Processed 55,500 / 60,104 sessions
Processed 60,104 / 60,104 sessions

=== FULL-SLATE TRAIN PERFORMANCE ===
Train NDCG@10: 0.454924
Sessions evaluated: 60104


In [16]:
# ============================================================
# TRAIN NDCG@10 — POPULARITY AND BM25 ONLY
# ============================================================

CHUNK = 500
rows = session_rows["train"]
totals = {"popularity": 0., "bm25": 0.}

bm25_col = next(c for c in df if "bm25_query_name" in c.lower())

for start in range(0, len(rows), CHUNK):

    idx = np.concatenate([
        ix for _, ix in rows[start:start + CHUNK]
    ])

    d = df.iloc[idx][
        ["session_id", "wid", "label", "candidate_pos", bm25_col]
    ].copy()

    wid = d["wid"].map(norm_id)
    keys = zip(d["session_id"].astype(int), wid)

    # Full-history popularity with leave-self-out
    pop = (
        wid.map(pop_product_users).fillna(0).to_numpy(float)
        + 0.001 * wid.map(pop_product_ord).fillna(0).to_numpy(float)
    )

    for j, key in enumerate(keys):
        pop[j] -= int(key in own_products)
        pop[j] -= 0.001 * own_orders.get(key, 0)

    scores = {
        "popularity": pop,
        "bm25": d[bm25_col].fillna(0).to_numpy(float)
    }

    for name, score in scores.items():
        totals[name] += (
            rank_metrics(d, score, seed=0)["ndcg@10"]
            * d["session_id"].nunique()
        )

n = len(rows)

print("=== TRAIN NDCG@10 ===")
for name, total in totals.items():
    print(f"{name}: {total / n:.6f}")

=== TRAIN NDCG@10 ===
popularity: 0.327746
bm25: 0.092929
